# Audio Deepfake Detector — training and inference walkthrough

This notebook is a **guide**, not a second training pipeline. The feature
extractor, training logic and evaluation live in ordinary Python files so the
Flask app and the experiments use exactly the same code.

**Pipeline:** ASVspoof 2019 LA → 16 kHz mono → 5 s chunks → 172 features →
ExtraTrees150 → Platt calibration → spoof/bonafide thresholds →
BONAFIDE / UNCERTAIN / SPOOF.

The shipped model is trained on speaker-disjoint dev data and checked on
unseen attacks A07–A19. Read `README.md` for the measured results.

## 1. Imports and paths

Run this cell first. It works whether the notebook is started from the project
folder or from another directory.

In [ ]:
import subprocess
import sys
from pathlib import Path

import numpy as np

PROJECT = Path.cwd()
if not (PROJECT / "config.py").exists():
    PROJECT = Path(r"F:\minor project\test\audio_deepfake_vscode")
sys.path.insert(0, str(PROJECT))

import audio_features as af
import config
import train_model as tm

print("Python :", sys.version.split()[0])
print("Project:", PROJECT)
print("Model  :", config.MODEL_PATH)
print("Exists :", config.MODEL_PATH.exists())

## 2. Dataset and splits

The protocol files are tiny, so this is fast. It confirms that `config.py`
points at the real dataset before any training starts.

In [ ]:
splits = {name: tm.read_split(name) for name in ("train", "dev", "eval")}
for name, rows in splits.items():
    bonafide = sum(row["label"] == 0 for row in rows)
    spoof = len(rows) - bonafide
    print(f"{name:5s}: {len(rows):>6,} files  bonafide={bonafide:>5,}  spoof={spoof:>6,}")

## 3. Train and compare models

Set `RUN_TRAINING = True` to run the full experiment:

* build/reuse feature caches in `benchmarks/`,
* compare feature sets and classifiers on **speaker-disjoint** dev data,
* calibrate the chosen model (Platt scaling),
* tune the spoof threshold for a 2% false-positive target,
* report unseen-attack metrics and save `models/audio_deepfake_model.joblib`.

The first full run takes roughly an hour because 50,000+ files are decoded.
Re-runs load the caches and finish in a few minutes.

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    subprocess.run(
        [sys.executable, "train_model.py", "--mode", "all"],
        cwd=PROJECT,
        check=True,
    )
else:
    print("Set RUN_TRAINING = True to rebuild features, compare models and save the bundle.")
    print("Tip: python train_model.py --mode compare --max-train-files 4000 is a faster experiment.")

## 4. Optional: format-robust final model

A model trained only on native 16 kHz FLAC files can mistake the high-frequency
roll-off of an MP3 for a spoof artefact. `augment_training.py` adds balanced
44.1 kHz and MP3 round-trip versions of the **training** files, and
`train_augmented.py` retrains and saves the production bundle.

This is the recommended final step if the app will accept real user uploads.

In [ ]:
RUN_AUGMENTATION = False

if RUN_AUGMENTATION:
    subprocess.run(
        [sys.executable, "augment_training.py", "--count", "2580",
         "--balanced", "--formats", "resample44,mp3"],
        cwd=PROJECT,
        check=True,
    )
    subprocess.run([sys.executable, "train_augmented.py"], cwd=PROJECT, check=True)
else:
    print("Set RUN_AUGMENTATION = True for the format-robust final model (~15 min).")

## 5. Inspect the shipped model

Everything below uses the model that the Flask app loads at startup.

In [ ]:
bundle = af.load_model(config.MODEL_PATH)
print("model      :", bundle.get("model_name"))
print("features   :", bundle.get("feature_set"), bundle.get("n_features"))
print("threshold  :", round(bundle.get("spoof_threshold", 0.5), 4))
print("bonafide   :", round(bundle.get("bonafide_threshold", 0.5), 4))
print("calibrated :", bundle.get("calibrator") is not None)
print("trained on :", bundle.get("trained_on_files"), "files /",
      bundle.get("trained_on_speakers"), "speakers")
if bundle.get("augmentation"):
    print("augmentation:", bundle["augmentation"])

validation = bundle.get("validation", {})
for name in ("dev_test", "eval_validation", "eval_holdout"):
    metrics = validation.get(name)
    if metrics:
        print(f"{name:16s} F1={metrics['f1']:.4f}  FPR={metrics['fpr']:.4f}  "
              f"FNR={metrics['fnr']:.4f}  AUC={metrics['roc_auc']:.4f}")

## 6. Predict two real dev files

One bonafide file and one spoof file, chosen automatically from the protocol.
The result includes the calibrated score, the decision band and the per-chunk
breakdown that the web app shows.

In [ ]:
dev = tm.read_split("dev")
bonafide = next(row for row in dev if row["label"] == 0)
spoof = next(row for row in dev if row["label"] == 1)
results = af.predict_audio_files([bonafide["path"], spoof["path"]], bundle)

for result in results:
    if result.get("error"):
        print(result["filename"], "ERROR", result["error"])
        continue
    print(f"{result['filename']:<24} {result['prediction']:<9} "
          f"score={result['spoof_score']:>6}%  confidence={result['confidence_label']:<6} "
          f"chunks={result['chunks_analysed']}")
    print("   ", result["reason"])
    for chunk in result["chunk_results"]:
        print(f"      {chunk['start_seconds']:>5}s  score={chunk['spoof_score']:>6}%  "
              f"weight={chunk['weight']}")

## 7. What to trust

* Dev accuracy is close to perfect because train and dev share attack types
  (A01–A06). The honest test is the **eval holdout**, which contains unseen
  attacks A07–A19.
* A17 and A18 are still hard. Per-attack recall is printed by
  `python final_evaluation.py`.
* Scores are calibrated with Platt scaling, but they are still model estimates.
  `UNCERTAIN` means the score lies between the bonafide and spoof thresholds,
  or the chunks disagree strongly.
* Phone recordings, heavy noise, music and non-speech audio are outside the
  ASVspoof distribution and may be unreliable.

## 8. Run the web app

```powershell
python app.py
```

Open http://127.0.0.1:5000. You can upload WAV/FLAC/MP3/OGG files, or press
**Start recording** to analyse your own voice. The browser turns the recording
into 16 kHz mono WAV and submits it through the same `/predict` endpoint, so the
model, calibration and thresholds are exactly the same as for file uploads.

`GET /health` reports the loaded model, its thresholds and its held-out metrics.
